# Collaborative Filtering Recommendation System

## Project Overview

The goal of this notebook is to develop the collaborative filtering component of the NutriCore recommendation workflow.

Unlike the previous recommendation approaches, collaborative filtering does not rely on recipe ingredients, nutritional information, or textual descriptions. Instead, it learns user preferences directly from historical interactions between users and recipes.

Using the Food.com reviews dataset, this notebook identifies patterns in user ratings to generate personalized recipe recommendations based on collective user behavior.

Three collaborative filtering techniques will be implemented and compared:

- User-Based Collaborative Filtering
- Item-Based Collaborative Filtering
- Matrix Factorization using Singular Value Decomposition (SVD)

The final collaborative filtering model developed in this notebook will later be integrated with the nutrition-based and content-based recommendation systems to create a Hybrid Recommendation System.

---

## Objectives

The objectives of this notebook are to:

- Load the processed recipe and review datasets.
- Construct a user–recipe interaction matrix.
- Analyze the characteristics of the interaction data.
- Implement User-Based Collaborative Filtering.
- Implement Item-Based Collaborative Filtering.
- Implement Matrix Factorization (SVD).
- Compare the recommendations generated by each approach.
- Prepare the collaborative filtering model for the final Hybrid Recommendation System.

In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "data").exists():
    raise FileNotFoundError("Open this notebook from the NutriCore repository or its notebooks directory.")

NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))
os.chdir(PROJECT_ROOT)

In [2]:
# Imports

import numpy as np
import pandas as pd
import joblib

from scipy.sparse.linalg import svds
from sklearn.neighbors import NearestNeighbors
from collections import defaultdict

# Display Settings
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

## Load Processed Datasets

The collaborative filtering models developed in this notebook rely on the processed datasets created during the previous stages of the project.

The recipe dataset contains the cleaned recipe information and engineered nutritional features, while the reviews dataset contains historical user ratings that represent interactions between users and recipes.

Loading the processed datasets ensures that all subsequent analyses and recommendation models operate on consistent and validated data.

In [3]:
# Load Processed Datasets

recipes_df = pd.read_parquet(
    "data/processed/recipes_processed.parquet"
)

reviews_df = pd.read_parquet(
    "data/processed/reviews_processed.parquet"
)

print(f"Recipes Shape : {recipes_df.shape}")
print(f"Reviews Shape : {reviews_df.shape}")

Recipes Shape : (302965, 16)
Reviews Shape : (788843, 8)


In [4]:
# Dataset Overview

print("Recipes Dataset")
display(recipes_df.head())

print("\nReviews Dataset")
display(reviews_df.head())

Recipes Dataset


,RecipeId,Name,Description,RecipeCategory,Keywords,RecipeIngredientParts,Calories,FatContent,CarbohydrateContent,ProteinContent,RecipeServings,DietType,CaloriesPerServing,ProteinPerServing,CarbsPerServing,FatPerServing
0,38.0,Low-Fat Berry Blue Frozen Dessert,Make and share this Low-Fat Berry Blue Frozen Dessert recipe from Food.com.,Frozen Desserts,"[Dessert, Low Protein, Low Cholesterol, Healthy, Free Of..., Summer, Weeknight, Freezer, Easy]","[blueberries, granulated sugar, vanilla yogurt, lemon juice]",170.9,2.5,37.1,3.2,4.0,Vegetarian,42.725000,0.800000,9.275000,0.625
1,39.0,Biryani,Make and share this Biryani recipe from Food.com.,Chicken Breast,"[Chicken Thigh & Leg, Chicken, Poultry, Meat, Asian, Indian, Weeknight, Stove Top]","[saffron, milk, hot green chili peppers, onions, garlic, clove, peppercorns, cardamom seed, cumin seed, poppy seed, mace, cilantro, mint leaf, fresh lemon juice, plain yogurt, boneless chicken, salt, ghee, onion, tomatoes, basmati rice, long-grain rice, raisins, cashews, eggs]",1110.7,58.8,84.4,63.4,6.0,Non-Vegetarian,185.116667,10.566667,14.066667,9.800
2,40.0,Best Lemonade,"This is from one of my first Good House Keeping cookbooks. You must use a *zester* in order to avoid getting any of that bitter rind, and when you zest the lemons, zest them onto some sugar from the recipe (the sugar will 'catch' all of the oils). I also advise you from personal experience to use only the best skinned lemons for the best flavor.",Beverages,"[Low Protein, Low Cholesterol, Healthy, Summer, < 60 Mins]","[sugar, lemons, rind of, lemon, zest of, fresh water, fresh lemon juice]",311.1,0.2,81.5,0.3,4.0,Vegetarian,77.775000,0.075000,20.375000,0.050
3,41.0,Carina's Tofu-Vegetable Kebabs,This dish is best prepared a day in advance to allow the ingredients to soak in the marinade overnight.,Soy/Tofu,"[Beans, Vegetable, Low Cholesterol, Weeknight, Broil/Grill, Oven]","[extra firm tofu, eggplant, zucchini, mushrooms, soy sauce, low sodium soy sauce, olive oil, maple syrup, honey, red wine vinegar, lemon juice, garlic cloves, mustard powder, black pepper]",536.1,24.0,64.2,29.3,2.0,Vegetarian,268.050000,14.650000,32.100000,12.000
4,42.0,Cabbage Soup,Make and share this Cabbage Soup recipe from Food.com.,Vegetable,"[Low Protein, Vegan, Low Cholesterol, Healthy, Winter, < 60 Mins, Easy]","[plain tomato juice, cabbage, onion, carrots, celery]",103.6,0.4,25.1,4.3,4.0,Vegetarian,25.900000,1.075000,6.275000,0.100



Reviews Dataset


,ReviewId,RecipeId,AuthorId,AuthorName,Rating,Review,DateSubmitted,DateModified
0,7,4384,1634,Bill Hilbrich,4,"I cut back on the mayo, and made up the difference with sour cream to adjust the stiffness of the dip.",2001-10-17 16:49:59+00:00,2001-10-17 16:49:59+00:00
1,9,4523,2046,Gay Gilmore ckpt,2,i think i did something wrong because i could taste the cornstarch in the finished product.,2000-02-25 09:00:00+00:00,2000-02-25 09:00:00+00:00
2,13,7435,1773,Malarkey Test,5,"easily the best i have ever had. juicy flavorful, not dry. the vegetables retain crispness as well, not bland at all!",2000-03-13 21:15:00+00:00,2000-03-13 21:15:00+00:00
3,14,44,2085,Tony Small,5,An excellent dish.,2000-03-28 12:51:00+00:00,2000-03-28 12:51:00+00:00
4,17,5221,2046,Gay Gilmore ckpt,4,"love it, but without the bean sprouts.",2000-05-08 11:08:00+00:00,2000-05-08 11:08:00+00:00


## Interaction Data

Collaborative filtering relies entirely on historical interactions between users and recipes.

Each review in the dataset represents an explicit interaction where a user has rated a particular recipe. Collectively, these interactions form the foundation of the recommendation system.

Before constructing recommendation models, it is important to understand the scale of the interaction data by answering the following questions:

- How many unique users are present?
- How many unique recipes have been rated?
- How many total interactions are available?
- What is the average number of ratings per user?
- What is the average number of ratings per recipe?

Understanding these characteristics provides insight into the density of the interaction matrix and helps identify potential challenges such as sparse data and cold-start scenarios.

In [5]:
# Interaction Statistics

n_users = reviews_df["AuthorId"].nunique()
n_recipes = reviews_df["RecipeId"].nunique()
n_interactions = len(reviews_df)

avg_ratings_per_user = (
    reviews_df
    .groupby("AuthorId")
    .size()
    .mean()
)

avg_ratings_per_recipe = (
    reviews_df
    .groupby("RecipeId")
    .size()
    .mean()
)

interaction_summary = pd.DataFrame({
    "Metric": [
        "Unique Users",
        "Unique Recipes",
        "Total Ratings",
        "Average Ratings per User",
        "Average Ratings per Recipe"
    ],
    "Value": [
        f"{n_users:,}",
        f"{n_recipes:,}",
        f"{n_interactions:,}",
        f"{avg_ratings_per_user:.2f}",
        f"{avg_ratings_per_recipe:.2f}"
    ]
})

interaction_summary

,Metric,Value
0,Unique Users,"170,039"
1,Unique Recipes,"155,842"
2,Total Ratings,"788,843"
3,Average Ratings per User,4.64
4,Average Ratings per Recipe,5.06


### Interaction Matrix Characteristics

A collaborative filtering model represents the dataset as a user–recipe interaction matrix.

- Rows represent users.
- Columns represent recipes.
- Each cell contains the rating given by a user to a recipe.

In practice, users rate only a small fraction of all available recipes. As a result, most entries in the interaction matrix remain empty.

This phenomenon is known as **matrix sparsity** and is one of the defining characteristics of recommendation systems.

Measuring matrix sparsity helps evaluate the complexity of the recommendation problem and motivates the use of techniques such as neighborhood-based collaborative filtering and matrix factorization.

In [6]:
# Matrix Sparsity

possible_interactions = n_users * n_recipes

sparsity = (
    1 - (n_interactions / possible_interactions)
) * 100

density = 100 - sparsity

matrix_summary = pd.DataFrame({
    "Metric": [
        "Possible User-Recipe Interactions",
        "Observed Ratings",
        "Matrix Density (%)",
        "Matrix Sparsity (%)"
    ],
    "Value": [
        f"{possible_interactions:,}",
        f"{n_interactions:,}",
        f"{density:.6f}",
        f"{sparsity:.6f}"
    ]
})

matrix_summary

,Metric,Value
0,Possible User-Recipe Interactions,"26,499,217,838"
1,Observed Ratings,"788,843"
2,Matrix Density (%),0.002977
3,Matrix Sparsity (%),99.997023


### Matrix Sparsity

Recommendation datasets are typically characterized by extremely sparse interaction matrices because each user interacts with only a very small subset of all available items.

A highly sparse matrix presents several challenges:

- Limited overlap between users makes similarity calculations more difficult.
- Many recipes receive only a few ratings.
- New users and new recipes suffer from the cold-start problem.
- Traditional machine learning techniques often struggle with sparse interaction data.

Collaborative filtering algorithms are specifically designed to learn meaningful recommendation patterns despite these challenges.

### Cold-Start Analysis

One of the most important challenges in collaborative filtering is the **cold-start problem**.

Collaborative filtering algorithms learn recommendation patterns from historical user interactions. When very little interaction data is available, generating reliable recommendations becomes significantly more difficult.

Cold-start scenarios generally occur in two forms:

- **New Users:** Users who have rated only a few recipes, providing insufficient information to learn their preferences.
- **New Recipes:** Recipes that have received very few ratings, making it difficult to estimate their relevance for other users.

Before building recommendation models, it is useful to examine how frequently these situations occur within the dataset.

Understanding the severity of the cold-start problem helps determine whether additional recommendation strategies, such as content-based filtering, are required. This is one of the primary motivations for developing a Hybrid Recommendation System later in this project.

In [7]:
# User Activity Distribution

user_activity = (
    reviews_df
    .groupby("AuthorId")
    .size()
    .rename("Ratings")
)

user_activity.describe().round(2)

count    170039.00
mean          4.64
std          40.50
min           1.00
25%           1.00
50%           1.00
75%           2.00
max        5172.00
Name: Ratings, dtype: float64

In [8]:
# Recipe Activity Distribution

recipe_activity = (
    reviews_df
    .groupby("RecipeId")
    .size()
    .rename("Ratings")
)

recipe_activity.describe().round(2)

count    155842.00
mean          5.06
std          20.32
min           1.00
25%           1.00
50%           2.00
75%           4.00
max        2892.00
Name: Ratings, dtype: float64

### Cold-Start Entities

To better understand the interaction data, users and recipes are grouped according to the number of ratings they have received.

For this analysis:

- Users with fewer than **5 ratings** are considered cold-start users.
- Recipes with fewer than **5 ratings** are considered cold-start recipes.

These thresholds are commonly used in recommendation system research to distinguish entities with limited interaction history from those with sufficient behavioral data.

In [9]:
# Cold-Start Statistics

cold_users = (user_activity < 5).sum()
cold_recipes = (recipe_activity < 5).sum()

cold_summary = pd.DataFrame({
    "Metric": [
        "Cold-Start Users (<5 Ratings)",
        "Cold-Start Recipes (<5 Ratings)"
    ],
    "Count": [
        cold_users,
        cold_recipes
    ],
    "Percentage": [
        cold_users / n_users * 100,
        cold_recipes / n_recipes * 100
    ]
})

cold_summary["Percentage"] = cold_summary["Percentage"].round(2)

cold_summary

,Metric,Count,Percentage
0,Cold-Start Users (<5 Ratings),152458,89.66
1,Cold-Start Recipes (<5 Ratings),118968,76.34


## Training Data Preparation

The previous analysis showed that the interaction matrix is extremely sparse, with a large proportion of users and recipes having very few recorded interactions.

Including these low-activity users and recipes in the collaborative filtering model introduces significant noise while providing little useful information for similarity calculations.

To improve recommendation quality and computational efficiency, a filtered interaction dataset is created exclusively for training the collaborative filtering models.

The filtering strategy adopted in this project is:

- Keep users who have provided at least **5 ratings**.
- Keep recipes that have received at least **5 ratings**.

It is important to note that this filtering step is applied **only during collaborative filtering model training**.

The complete recipe catalogue and review dataset are preserved for the overall application and will later be utilized by the Hybrid Recommendation System. Consequently, recipes excluded from collaborative filtering can still be recommended through the nutrition-based and content-based recommendation modules.

In [10]:
# Active Users

active_users = user_activity[user_activity >= 5].index

active_recipes = recipe_activity[recipe_activity >= 5].index


In [11]:
# Filter Interaction Dataset

cf_reviews_df = reviews_df[
    (reviews_df["AuthorId"].isin(active_users)) &
    (reviews_df["RecipeId"].isin(active_recipes))
].copy()

print(f"Filtered Reviews : {len(cf_reviews_df):,}")

Filtered Reviews : 420,570


### Training Dataset Summary

After filtering inactive users and rarely rated recipes, the collaborative filtering training dataset becomes substantially smaller while retaining the most informative user–recipe interactions.

The following summary compares the filtered dataset with the original interaction data and illustrates the reduction in users, recipes, and ratings prior to model construction.

In [12]:
# Filtered Dataset Summary

filtered_users = cf_reviews_df["AuthorId"].nunique()
filtered_recipes = cf_reviews_df["RecipeId"].nunique()
filtered_ratings = len(cf_reviews_df)

summary = pd.DataFrame({
    "Metric": [
        "Active Users",
        "Active Recipes",
        "Remaining Ratings"
    ],
    "Value": [
        f"{filtered_users:,}",
        f"{filtered_recipes:,}",
        f"{filtered_ratings:,}"
    ]
})

summary

,Metric,Value
0,Active Users,"17,546"
1,Active Recipes,"36,632"
2,Remaining Ratings,"420,570"


In [13]:
# Matrix Sparsity After Filtering

possible_interactions = filtered_users * filtered_recipes

density = (
    filtered_ratings /
    possible_interactions
) * 100

sparsity = 100 - density

filtered_matrix = pd.DataFrame({
    "Metric": [
        "Possible Interactions",
        "Observed Ratings",
        "Matrix Density (%)",
        "Matrix Sparsity (%)"
    ],
    "Value": [
        f"{possible_interactions:,}",
        f"{filtered_ratings:,}",
        f"{density:.6f}",
        f"{sparsity:.6f}"
    ]
})

filtered_matrix

,Metric,Value
0,Possible Interactions,"642,745,072"
1,Observed Ratings,"420,570"
2,Matrix Density (%),0.065433
3,Matrix Sparsity (%),99.934567


## Neighborhood-Based Collaborative Filtering

Collaborative filtering algorithms require the interaction data to be represented as a matrix.

In this matrix:

- Each row represents a unique user.
- Each column represents a unique recipe.
- Each cell contains the rating assigned by a user to a recipe.

If a user has not rated a particular recipe, the corresponding entry is assigned a value of zero.

This matrix serves as the foundation for all collaborative filtering techniques implemented in this notebook.

The same interaction matrix will subsequently be used to develop:

- User-Based Collaborative Filtering
- Item-Based Collaborative Filtering
- Matrix Factorization (SVD)

Constructing a single interaction matrix ensures consistency across all recommendation approaches and enables direct comparison of their results.

In [14]:
# User-Recipe Interaction Matrix

interaction_matrix = cf_reviews_df.pivot_table(
    index="AuthorId",
    columns="RecipeId",
    values="Rating",
    fill_value=0
)

print(f"Interaction Matrix Shape : {interaction_matrix.shape}")

interaction_matrix.iloc[:5, :10]

Interaction Matrix Shape : (17546, 36632)


RecipeId,40,42,44,54,56,58,62,71,72,76
AuthorId,,,,,,,,,,
1533,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1535,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1634,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1676,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1792,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### Interaction Matrix

The interaction matrix contains one row for each active user and one column for each active recipe.

Each cell represents the rating given by a user to a particular recipe.

- A positive value indicates that the user has rated the recipe.
- A value of **0** indicates that no interaction exists between the user and the recipe.

It is important to note that a value of **0 does not represent a poor rating**. Instead, it simply means that the user has never rated that recipe.

When displaying only the first few rows and columns of the interaction matrix, it is common to observe many zeros. This occurs because the displayed users may have rated entirely different recipes than those shown in the displayed columns.

For example, if the first five users have not interacted with the first ten recipes, every displayed value will be zero even though those users may have rated many other recipes elsewhere in the matrix.

This sparse representation is expected in collaborative filtering problems, where each user typically interacts with only a very small subset of the available items.

### Interaction Matrix Verification

Before applying collaborative filtering algorithms, it is important to verify that the interaction matrix has been constructed correctly.

The matrix should satisfy the following conditions:

- Rows correspond to unique users.
- Columns correspond to unique recipes.
- Cell values represent user ratings.
- Missing interactions are represented by zeros.

The dimensions of the interaction matrix should match the number of active users and active recipes identified during the filtering stage.

In [15]:
# Example User Interactions

sample_user = 37449

print(f"Sample User ID : {sample_user}\n")

display(
    interaction_matrix.loc[sample_user][
        interaction_matrix.loc[sample_user] > 0
    ]
)

Sample User ID : 37449



RecipeId
548       5.0
719       5.0
860       5.0
1460      5.0
1780      4.0
         ... 
505571    5.0
505620    5.0
505784    5.0
506822    5.0
507924    5.0
Name: 37449, Length: 2179, dtype: float64

### Similarity Search Strategy

A traditional implementation of User-Based and Item-Based Collaborative Filtering computes the cosine similarity between every pair of users or recipes.

Although straightforward, this approach becomes computationally expensive for large datasets because it requires storing an entire similarity matrix.

For this project:

- Active Users: approximately 17,500
- Active Recipes: approximately 36,600

Constructing complete similarity matrices for datasets of this size would require computing hundreds of millions to billions of similarity values, resulting in substantial memory consumption and unnecessary computation.

Instead, this project employs the **Nearest Neighbors** algorithm using cosine distance.

Rather than calculating similarities between every possible pair, the algorithm identifies only the **K most similar neighbors** for a given user or recipe.

This approach offers several advantages:

- Significantly lower memory requirements.
- Faster recommendation generation.
- Better scalability for larger datasets.
- Produces recommendations equivalent to neighborhood-based collaborative filtering while avoiding the need to store a full similarity matrix.

This design closely reflects the techniques used in modern recommendation systems, where scalable nearest-neighbor search is preferred over exhaustive similarity computation.

### Nearest Neighbors Model

Neighborhood-based collaborative filtering identifies users or recipes with similar interaction patterns.

Rather than computing similarities between every possible pair of users or recipes, this project uses the **Nearest Neighbors** algorithm with the cosine distance metric.

Given a target user (or recipe), the algorithm efficiently identifies the **K most similar neighbors**, significantly reducing both memory usage and computational cost compared to constructing a complete similarity matrix.

The trained nearest neighbors model will subsequently be used for:

- User-Based Collaborative Filtering
- Item-Based Collaborative Filtering

The only difference between these approaches is the orientation of the interaction matrix:

- **User-Based Collaborative Filtering:** Users are treated as observations.
- **Item-Based Collaborative Filtering:** Recipes are treated as observations by transposing the interaction matrix.

In [16]:
# Build User-Based Nearest Neighbors Model

user_knn = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_neighbors=11,      # 10 neighbours + the user itself
    n_jobs=-1
)

user_knn.fit(interaction_matrix)

print("User-Based Nearest Neighbors model trained successfully.")

User-Based Nearest Neighbors model trained successfully.


### Similar Users

Before generating recommendations, it is useful to verify that the nearest neighbors model is able to identify users with similar rating patterns.

For a selected user, the model returns the most similar users according to cosine distance.

Since every user is perfectly similar to themselves, the first returned neighbor is always the queried user. Therefore, this result is excluded when generating recommendations.

In [18]:
# Find Similar Users

sample_user = 37449

distances, indices = user_knn.kneighbors(
    interaction_matrix.loc[[sample_user]]
)

similar_users = pd.DataFrame({
    "AuthorId": interaction_matrix.index[indices.flatten()],
    "CosineDistance": distances.flatten()
})

# Remove Query User
similar_users = similar_users.iloc[1:].reset_index(drop=True)

similar_users["CosineSimilarity"] = (
    1 - similar_users["CosineDistance"]
)

similar_users = similar_users[
    ["AuthorId", "CosineSimilarity", "CosineDistance"]
]

similar_users

,AuthorId,CosineSimilarity,CosineDistance
0,128473,0.115943,0.884057
1,80353,0.115164,0.884836
2,226863,0.114853,0.885147
3,542159,0.113072,0.886928
4,166642,0.111565,0.888435
5,305531,0.105825,0.894175
6,593927,0.102745,0.897255
7,383346,0.101762,0.898238
8,678366,0.101270,0.898730
9,424680,0.097756,0.902244


### Cosine Distance

The Nearest Neighbors model returns **cosine distance**, which measures the dissimilarity between two users based on their rating patterns.

Cosine similarity can be obtained as:

**Cosine Similarity = 1 − Cosine Distance**

A smaller cosine distance indicates that two users have more similar rating behaviour.

It is important to note that recommendation datasets are typically **extremely sparse**, meaning that users rate only a very small fraction of the available items.

As a result, even the nearest neighbors may have relatively large cosine distances because they share only a limited number of common interactions.

Therefore, collaborative filtering focuses on identifying the **most similar users relative to one another**, rather than expecting very small distance values in absolute terms.

### User-Based Recommendations

User-Based Collaborative Filtering recommends recipes by identifying users with similar historical rating behaviour.

Unlike content-based filtering, which relies on recipe characteristics, collaborative filtering learns entirely from user interactions. The underlying assumption is that users who exhibited similar preferences in the past are likely to enjoy similar recipes in the future.

The recommendation process consists of the following steps:

1. Identify the target user.
2. Retrieve the **K** most similar users using the trained Nearest Neighbors model.
3. Collect recipes rated by these similar users.
4. Exclude recipes already rated by the target user.
5. Estimate a recommendation score using similarity-weighted mean-centered collaborative filtering.
6. Apply **significance weighting** to reward recipes supported by a larger number of similar users.
7. Rank candidate recipes according to the final recommendation score.
8. Return the highest-ranked recommendations.

This implementation uses **mean-centered collaborative filtering**, where each neighbor's rating is adjusted relative to their own average rating before contributing to the recommendation score.

To improve recommendation reliability, **significance weighting** is applied. Recipes supported by a larger number of similar users receive greater confidence, while recipes rated by only one or two neighbors are naturally penalized.

The recommendation score is computed as

$$
\hat{r}_{ui} =
\bar{r}_{u} +
\frac{
\sum_{v \in N(u)} s_{uv} (r_{vi} - \bar{r}_{v})
}{
\sum_{v \in N(u)} |s_{uv}|
}
$$

The final recommendation score is then calculated as

$$
\text{Final Recommendation Score} =
\hat{r}_{ui} \times \frac{n_i}{n_i + k}
$$

where

- $\hat{r}_{ui}$ is the mean-centered collaborative filtering score.
- $n_i$ is the number of similar users who rated recipe $i$.
- $k$ is the significance weighting parameter controlling the influence of neighbor support.

In this implementation, **k = 10**. As the number of supporting neighbors increases, the significance weight approaches 1, allowing highly supported recommendations to retain their full score while reducing the influence of recommendations based on very limited collaborative evidence.

In [19]:
### User-Based Recommendations

def recommend_user_based(
    user_id,
    interaction_matrix,
    user_knn,
    recipes_df,
    n_neighbors=50,
    n_recommendations=10,
    significance_k=10
):
    """
    Generate recipe recommendations using
    mean-centered User-Based Collaborative Filtering.

    The RecommendationScore is used to rank recipes
    within the collaborative filtering model.
    """

    # Validate User

    if user_id not in interaction_matrix.index:
        raise ValueError(
            f"User {user_id} not found."
        )

    # Find Similar Users

    distances, indices = user_knn.kneighbors(
        interaction_matrix.loc[[user_id]],
        n_neighbors=n_neighbors + 1
    )

    neighbor_ids = interaction_matrix.index[
        indices.flatten()[1:]
    ]

    similarities = 1 - distances.flatten()[1:]

    # Target User Information

    target_ratings = interaction_matrix.loc[user_id]

    rated_recipes = set(
        target_ratings[target_ratings > 0].index
    )

    target_mean = (
        target_ratings[target_ratings > 0].mean()
    )

    # Aggregate Neighbor Contributions

    weighted_deviations = defaultdict(float)
    similarity_sums = defaultdict(float)

    neighbor_counts = defaultdict(int)
    rating_sums = defaultdict(float)

    for neighbor_id, similarity in zip(
        neighbor_ids,
        similarities
    ):

        neighbor_ratings = interaction_matrix.loc[
            neighbor_id
        ]

        neighbor_mean = (
            neighbor_ratings[
                neighbor_ratings > 0
            ].mean()
        )

        for recipe_id, rating in neighbor_ratings.items():

            if rating == 0:
                continue

            if recipe_id in rated_recipes:
                continue

            deviation = (
                rating - neighbor_mean
            )

            weighted_deviations[recipe_id] += (
                deviation * similarity
            )

            similarity_sums[recipe_id] += abs(
                similarity
            )

            neighbor_counts[recipe_id] += 1

            rating_sums[recipe_id] += rating

    # Compute Recommendation Scores

    recommendations = []

    for recipe_id in weighted_deviations:

        # Mean-centered collaborative filtering score
        recommendation_score = (
            target_mean
            +
            weighted_deviations[recipe_id]
            /
            similarity_sums[recipe_id]
        )

        # Significance weighting
        significance_weight = (
            neighbor_counts[recipe_id]
            /
            (
                neighbor_counts[recipe_id]
                + significance_k
            )
        )

        final_score = (
            recommendation_score
            * significance_weight
        )

        average_neighbor_rating = (
            rating_sums[recipe_id]
            /
            neighbor_counts[recipe_id]
        )

        recommendations.append(
            (
                recipe_id,
                final_score,
                neighbor_counts[recipe_id],
                average_neighbor_rating
            )
        )

    # Rank Recommendations

    recommendations = sorted(
        recommendations,
        key=lambda x: (
            x[1],   # Recommendation Score
            x[2]    # Neighbor Count
        ),
        reverse=True
    )[:n_recommendations]

    # Convert to DataFrame

    recommendation_df = pd.DataFrame(
        recommendations,
        columns=[
            "RecipeId",
            "RecommendationScore",
            "NeighborCount",
            "AverageNeighborRating"
        ]
    )

    # Merge Recipe Metadata

    recommendation_df = recommendation_df.merge(
        recipes_df,
        on="RecipeId",
        how="left"
    )

    # Normalize Recommendation Score

    max_score = recommendation_df[
        "RecommendationScore"
    ].max()

    if max_score > 0:

        recommendation_df[
            "NormalizedScore"
        ] = (
            recommendation_df[
                "RecommendationScore"
            ]
            /
            max_score
        )

    else:

        recommendation_df[
            "NormalizedScore"
        ] = 0.0

    # Format Output

    recommendation_df[
        [
            "RecommendationScore",
            "NormalizedScore",
            "AverageNeighborRating"
        ]
    ] = recommendation_df[
        [
            "RecommendationScore",
            "NormalizedScore",
            "AverageNeighborRating"
        ]
    ].round(3)

    recommendation_df = (
        recommendation_df
        .sort_values(
            by=[
                "RecommendationScore",
                "NeighborCount"
            ],
            ascending=False
        )
        .reset_index(drop=True)
    )

    return recommendation_df

### User-Based Check

Recommendations are generated for a representative user selected from the filtered interaction dataset.

The returned recommendations should satisfy the following conditions:

- The target user exists in the interaction matrix.
- Previously rated recipes are excluded.
- Recommendation scores are computed using similarity-weighted mean-centered collaborative filtering.
- Significance weighting is applied to favour recipes supported by a larger number of similar users.
- Candidate recipes are ranked according to the final recommendation score.
- Neighbor statistics are returned to provide additional insight into the collaborative evidence supporting each recommendation.
- Recipe metadata is successfully retrieved from the processed recipe dataset.

In [20]:
# Example Recommendation

# Representative user selected for demonstration
sample_user = 37449


# Display User's Top Rated Recipes

sample_user_ratings = (
    interaction_matrix
    .loc[sample_user]
)

sample_user_recipes = (
    sample_user_ratings[
        sample_user_ratings > 0
    ]
    .sort_values(ascending=False)
    .head(5)
    .reset_index()
)

sample_user_recipes.columns = [
    "RecipeId",
    "Rating"
]

sample_user_recipes = sample_user_recipes.merge(
    recipes_df[
        [
            "RecipeId",
            "Name"
        ]
    ],
    on="RecipeId",
    how="left"
)


print("=" * 60)
print("USER'S SAMPLE RATED RECIPES")
print("=" * 60)

display(
    sample_user_recipes
)


# Generate User-Based Recommendations

recommendations = recommend_user_based(
    user_id=sample_user,
    interaction_matrix=interaction_matrix,
    user_knn=user_knn,
    recipes_df=recipes_df,
    n_neighbors=50,
    n_recommendations=10,
    significance_k=10
)


print("=" * 60)
print("USER-BASED RECOMMENDATIONS")
print("=" * 60)

recommendations[
    [
        "RecipeId",
        "Name",
        "RecommendationScore",
        "NormalizedScore",
        "NeighborCount",
        "AverageNeighborRating"
    ]
]

USER'S SAMPLE RATED RECIPES


,RecipeId,Rating,Name
0,548,5.0,Grilled Bourbon Salmon
1,281409,5.0,Absolutely the Best Southern Sweet Tea
2,274348,5.0,John Besh's Creamy Polenta With Mascarpone Cheese
3,274237,5.0,Giant Bittersweet Chocolate Toffee Cookies
4,274204,5.0,Iceberg Wedges With Creamy Blue Cheese Dressing


USER-BASED RECOMMENDATIONS


,RecipeId,Name,RecommendationScore,NormalizedScore,NeighborCount,AverageNeighborRating
0,73033,Allrighty then Scrambled Eggs - Paula Deen,3.383,1.000,20,5.000
1,185342,Weight Watchers Parmesan Chicken Cutlets,3.070,0.907,15,5.000
2,113983,Classic Saffron Rice,3.065,0.906,16,4.938
3,132765,Microwave Lemon Garlic Broccoli,3.019,0.892,17,4.765
4,47077,Tartar Sauce,2.910,0.860,14,4.929
5,77397,Kittencal's 5-Minute Cinnamon Flop Brunch Cake,2.878,0.851,15,4.733
6,366061,Marinated Tomatoes for Your Salad (Or As a Side Dish!),2.870,0.848,13,5.000
7,350531,Homemade Oven Chips,2.852,0.843,13,5.000
8,272233,Down on the Farm - Fried Ham Sandwich - Longmeadow,2.842,0.840,13,5.000
9,148215,It's a Tuna Salad Wrap!,2.817,0.833,13,4.923


### Item-Based Recommendations

Item-Based Collaborative Filtering recommends recipes by identifying recipes that are similar to those a user has previously liked or highly rated.

Unlike User-Based Collaborative Filtering, which searches for users with similar rating behaviour, Item-Based Collaborative Filtering focuses on relationships between recipes. The underlying assumption is that recipes frequently liked by the same users are likely to be similar and therefore suitable recommendations.

Because recipe similarities tend to remain relatively stable over time, Item-Based Collaborative Filtering is generally more scalable than User-Based Collaborative Filtering in large recommendation systems.

In this notebook, recipe similarities are learned from historical user ratings using cosine similarity and the K-Nearest Neighbors (KNN) algorithm. Recommendations are generated by combining the similarities of recipes previously rated by the target user with the corresponding user ratings.

The implementation consists of the following stages:

1. Construct the recipe-user interaction matrix.
2. Train an Item-Based K-Nearest Neighbors model using cosine similarity.
3. Retrieve similar recipes for every recipe rated by the target user.
4. Compute a weighted recommendation score based on recipe similarities and user ratings.
5. Exclude recipes already rated by the target user.
6. Rank candidate recipes according to their recommendation score.
7. Return the highest-ranked recommendations.

In [21]:
# Recipe-User Interaction Matrix

item_matrix = interaction_matrix.T

print("=" * 60)
print("ITEM INTERACTION MATRIX")
print("=" * 60)

print(f"Shape : {item_matrix.shape}")

item_matrix.iloc[:5, :10]

ITEM INTERACTION MATRIX
Shape : (36632, 17546)


AuthorId,1533,1535,1634,1676,1792,1891,1962,2046,2059,2148
RecipeId,,,,,,,,,,
40,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
42,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
44,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
54,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
56,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [22]:
# Train Item-Based KNN

item_knn = NearestNeighbors(
    metric="cosine",
    algorithm="brute"
)

item_knn.fit(item_matrix)

print("=" * 60)
print("ITEM-BASED KNN TRAINED")
print("=" * 60)

print(f"Recipes Indexed : {item_matrix.shape[0]}")

ITEM-BASED KNN TRAINED
Recipes Indexed : 36632


### Similar Recipes

Before generating recommendations, it is useful to verify that the Item-Based K-Nearest Neighbors model has learned meaningful relationships between recipes.

Given a recipe, the model retrieves the **K** most similar recipes based on cosine similarity of their user-rating vectors.

Recipes that are frequently rated similarly by the same users will have higher cosine similarity and are therefore considered better recommendation candidates.

The Nearest Neighbors model returns cosine distance, which is converted to cosine similarity using

$$
\text{Cosine Similarity} = 1 - \text{Cosine Distance}
$$

Higher similarity values indicate stronger relationships between recipes.

In [23]:
# Find Similar Recipes

def find_similar_recipes(
    recipe_id,
    item_matrix,
    item_knn,
    recipes_df,
    n_neighbors=10
):
    """
    Retrieve recipes most similar to
    the specified recipe.
    """

    # Validate Recipe

    if recipe_id not in item_matrix.index:
        raise ValueError(
            f"Recipe {recipe_id} not found."
        )

    # Retrieve Similar Recipes

    distances, indices = item_knn.kneighbors(
        item_matrix.loc[[recipe_id]],
        n_neighbors=n_neighbors + 1
    )

    similar_recipe_ids = item_matrix.index[
        indices.flatten()[1:]
    ]

    similarities = (
        1 - distances.flatten()[1:]
    )

    # Build Results

    similar_df = pd.DataFrame({
        "RecipeId": similar_recipe_ids,
        "Similarity": similarities
    })

    similar_df = similar_df.merge(
        recipes_df[
            [
                "RecipeId",
                "Name",
                "RecipeCategory",
                "DietType"
            ]
        ],
        on="RecipeId",
        how="left"
    )

    similar_df = similar_df[
        [
            "RecipeId",
            "Name",
            "RecipeCategory",
            "DietType",
            "Similarity"
        ]
    ]

    similar_df["Similarity"] = (
        similar_df["Similarity"]
        .round(3)
    )

    return similar_df

### Similar Recipe Check

To verify the Item-Based Collaborative Filtering model, a sample recipe is selected and its most similar recipes are retrieved.

The retrieved recipes should:

- Exclude the query recipe itself.
- Exhibit high cosine similarity.
- Share similar historical rating patterns with the query recipe.
- Demonstrate that the Item-Based KNN model has learned meaningful recipe relationships.

In [24]:
# Test Similar Recipes

sample_recipe = 45809

print(f"Recipe ID   : {sample_recipe}")

recipe_name = recipes_df.loc[
    recipes_df["RecipeId"] == sample_recipe,
    "Name"
].iat[0]

print(f"Recipe Name : {recipe_name}")

similar_recipes = find_similar_recipes(
    recipe_id=sample_recipe,
    item_matrix=item_matrix,
    item_knn=item_knn,
    recipes_df=recipes_df,
    n_neighbors=10
)

similar_recipes

Recipe ID   : 45809
Recipe Name : Bourbon Chicken


,RecipeId,Name,RecipeCategory,DietType,Similarity
0,39087,Creamy Cajun Chicken Pasta,Chicken Breast,Non-Vegetarian,0.208
1,27208,To Die for Crock Pot Roast,One Dish Meal,Non-Vegetarian,0.199
2,89204,Crock-Pot Chicken With Black Beans &amp; Cream Cheese,One Dish Meal,Non-Vegetarian,0.188
3,32204,&quot;Whatever Floats Your Boat&quot; Brownies!,Bar Cookie,Eggitarian,0.175
4,68955,Japanese Mum's Chicken,Chicken Thigh & Leg,Non-Vegetarian,0.169
5,82102,Kittencal's Moist Cheddar-Garlic Oven Fried Chicken Breast,Chicken Breast,Non-Vegetarian,0.169
6,77397,Kittencal's 5-Minute Cinnamon Flop Brunch Cake,Breads,Eggitarian,0.164
7,65768,Chicken Lazone,Chicken,Non-Vegetarian,0.159
8,63689,My Family's Favorite Sloppy Joes (Pizza Joes),Lunch/Snacks,Non-Vegetarian,0.158
9,129926,Simply Sour Cream Chicken Enchiladas,Chicken Breast,Non-Vegetarian,0.156


### Item-Based Recommendation Generation

Item-Based Collaborative Filtering recommends recipes by identifying recipes that are similar to those previously rated by the target user.

Unlike User-Based Collaborative Filtering, which relies on similarities between users, Item-Based Collaborative Filtering focuses on relationships between recipes. The underlying assumption is that recipes receiving similar rating patterns from users are likely to appeal to users with similar preferences.

For each recipe previously rated by the target user, the Item-Based K-Nearest Neighbors model retrieves its most similar neighboring recipes. These neighboring recipes are weighted according to both their similarity and the user's original rating to compute a recommendation score.

Since highly active users may have rated thousands of recipes, evaluating every rated recipe would require thousands of nearest-neighbor searches during recommendation. To improve efficiency while maintaining recommendation quality, this implementation considers only the user's highest-rated recipes as the source of recommendations. This significantly reduces computation time while preserving the strongest indicators of user preference.

The recommendation process consists of the following steps:

1. Retrieve the target user's highest-rated recipes.
2. Find the most similar neighboring recipes for each rated recipe.
3. Exclude recipes already rated by the target user.
4. Compute similarity-weighted recommendation scores.
5. Aggregate scores from multiple source recipes.
6. Rank candidate recipes according to their recommendation score.
7. Return the highest-ranked recommendations.

In [25]:
### Item-Based Recommendations

def recommend_item_based(
    user_id,
    interaction_matrix,
    item_matrix,
    item_knn,
    recipes_df,
    n_neighbors=20,
    max_source_recipes=100,
    n_recommendations=10
):
    """
    Generate recipe recommendations using
    Item-Based Collaborative Filtering.
    """

    # Validate User

    if user_id not in interaction_matrix.index:
        raise ValueError(
            f"User {user_id} not found."
        )

    # Retrieve User Ratings

    user_ratings = interaction_matrix.loc[user_id]

    rated_recipes = (
        user_ratings[user_ratings > 0]
        .sort_values(ascending=False)
        .head(max_source_recipes)
    )

    rated_recipe_ids = set(rated_recipes.index)

    # Aggregate Recommendation Scores

    weighted_scores = defaultdict(float)

    contributing_recipes = defaultdict(int)

    # Process Each Rated Recipe

    for recipe_id, user_rating in rated_recipes.items():

        if recipe_id not in item_matrix.index:
            continue

        distances, indices = item_knn.kneighbors(
            item_matrix.loc[[recipe_id]],
            n_neighbors=n_neighbors + 1
        )

        similar_recipe_ids = item_matrix.index[
            indices.flatten()[1:]
        ]

        similarities = (
            1 - distances.flatten()[1:]
        )

        for similar_recipe, similarity in zip(
            similar_recipe_ids,
            similarities
        ):

            if similar_recipe in rated_recipe_ids:
                continue

            weighted_scores[
                similar_recipe
            ] += (
                similarity * user_rating
            )

            contributing_recipes[
                similar_recipe
            ] += 1

    # Compute Recommendation Scores

    recommendations = []

    for recipe_id in weighted_scores:

        recommendation_score = (
            weighted_scores[recipe_id]
        )

        recommendations.append(
            (
                recipe_id,
                recommendation_score,
                contributing_recipes[recipe_id]
            )
        )

    # Rank Recommendations

    recommendations = sorted(
        recommendations,
        key=lambda x: (
            x[1],
            x[2]
        ),
        reverse=True
    )[:n_recommendations]

    # Convert to DataFrame

    recommendation_df = pd.DataFrame(
        recommendations,
        columns=[
            "RecipeId",
            "RecommendationScore",
            "ContributingRecipes"
        ]
    )

    # Merge Recipe Metadata

    recommendation_df = recommendation_df.merge(
        recipes_df,
        on="RecipeId",
        how="left"
    )

    # Normalize Recommendation Score

    max_score = recommendation_df[
        "RecommendationScore"
    ].max()

    if max_score > 0:

        recommendation_df[
            "NormalizedScore"
        ] = (
            recommendation_df[
                "RecommendationScore"
            ]
            /
            max_score
        )

    else:

        recommendation_df[
            "NormalizedScore"
        ] = 0.0

    # Format Output

    recommendation_df[
        [
            "RecommendationScore",
            "NormalizedScore"
        ]
    ] = recommendation_df[
        [
            "RecommendationScore",
            "NormalizedScore"
        ]
    ].round(3)

    recommendation_df = (
        recommendation_df
        .sort_values(
            by=[
                "RecommendationScore",
                "ContributingRecipes"
            ],
            ascending=False
        )
        .reset_index(drop=True)
    )

    return recommendation_df

### Item-Based Check

Recommendations are generated for the same representative user used during User-Based Collaborative Filtering.

The returned recommendations should satisfy the following conditions:

- Previously rated recipes are excluded.
- Recommendations are generated from similar recipes rather than similar users.
- Recommendation scores reflect similarity-weighted user preferences.
- Recipes influenced by multiple previously rated recipes receive stronger collaborative evidence.

In [26]:
# Example Recommendation

sample_user = 37449

# Display User's Top Rated Recipes

sample_user_ratings = (
    interaction_matrix
    .loc[sample_user]
)

sample_user_recipes = (
    sample_user_ratings[
        sample_user_ratings > 0
    ]
    .sort_values(ascending=False)
    .head(5)
    .reset_index()
)

sample_user_recipes.columns = [
    "RecipeId",
    "Rating"
]

sample_user_recipes = sample_user_recipes.merge(
    recipes_df[
        [
            "RecipeId",
            "Name"
        ]
    ],
    on="RecipeId",
    how="left"
)

print("=" * 60)
print("USER'S SAMPLE LIKED RECIPES")
print("=" * 60)

display(sample_user_recipes)


# Generate Item-Based Recommendations

recommendations = recommend_item_based(
    user_id=sample_user,
    interaction_matrix=interaction_matrix,
    item_matrix=item_matrix,
    item_knn=item_knn,
    recipes_df=recipes_df,
    n_neighbors=20,
    max_source_recipes=50,
    n_recommendations=10
)


print("=" * 60)
print("ITEM-BASED RECOMMENDATIONS")
print("=" * 60)

recommendations[
    [
        "RecipeId",
        "Name",
        "RecommendationScore",
        "NormalizedScore",
        "ContributingRecipes"
    ]
]

USER'S SAMPLE LIKED RECIPES


,RecipeId,Rating,Name
0,548,5.0,Grilled Bourbon Salmon
1,281409,5.0,Absolutely the Best Southern Sweet Tea
2,274348,5.0,John Besh's Creamy Polenta With Mascarpone Cheese
3,274237,5.0,Giant Bittersweet Chocolate Toffee Cookies
4,274204,5.0,Iceberg Wedges With Creamy Blue Cheese Dressing


ITEM-BASED RECOMMENDATIONS


,RecipeId,Name,RecommendationScore,NormalizedScore,ContributingRecipes
0,25047,French Apple Pie,43.528,1.000,22
1,86466,Low Carb Kfc Coleslaw,13.974,0.321,8
2,343115,Mocha Java,13.639,0.313,6
3,301745,Three Mushrooms Crostini,11.212,0.258,5
4,333573,Cantaloupe Smoothie,10.830,0.249,5
5,45920,Church's Fried Chicken,9.713,0.223,6
6,296105,Smashed Loaded Potatoes (Low Fat),9.418,0.216,4
7,290479,Feeling Down,9.268,0.213,4
8,289425,Maytag Blue Cheese Salad Dressing,9.175,0.211,5
9,299853,Brown Sugar Coffee Cake Cappuccino,8.796,0.202,5


## Matrix Factorization with SVD


### SVD Approach

The previous collaborative filtering approaches used memory-based techniques:

- User-Based Collaborative Filtering:
  Finds users with similar rating behavior and recommends recipes based on their preferences.

- Item-Based Collaborative Filtering:
  Finds recipes with similar interaction patterns and recommends recipes similar to what a user already likes.

Although effective, memory-based methods have limitations when dealing with highly sparse datasets because they rely on direct similarity calculations.

Matrix Factorization approaches such as SVD address this by learning lower-dimensional latent representations of users and recipes.

Instead of comparing users or recipes directly, SVD decomposes the interaction matrix into hidden factors that capture underlying preference patterns.

The learned representations are then used to predict missing user-recipe ratings and generate personalized recommendations.

In [27]:
# Import SVD Libraries

from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from surprise import accuracy

import pickle

In [28]:
# Prepare Dataset for SVD

svd_df = cf_reviews_df[
    [
        "AuthorId",
        "RecipeId",
        "Rating"
    ]
].copy()

svd_df.head()

,AuthorId,RecipeId,Rating
1,2046,4523,2
4,2046,5221,4
6,2046,517,5
7,2046,3431,5
8,2369,5221,5


In [29]:
# SVD Dataset Summary

print("=" * 60)
print("SVD TRAINING DATA")
print("=" * 60)

print(f"Total Ratings  : {len(svd_df):,}")
print(f"Unique Users   : {svd_df['AuthorId'].nunique():,}")
print(f"Unique Recipes : {svd_df['RecipeId'].nunique():,}")

print(
    f"Rating Range   : "
    f"{svd_df['Rating'].min()} - {svd_df['Rating'].max()}"
)

SVD TRAINING DATA
Total Ratings  : 420,570
Unique Users   : 17,546
Unique Recipes : 36,632
Rating Range   : 0 - 5


### Preparing SVD Data

The SVD implementation requires user-recipe-rating interactions instead of the complete sparse interaction matrix.

Only observed ratings are used during training. The model learns latent user and recipe representations from these interactions and uses them to estimate ratings for unseen recipes.

In [30]:
# Create Surprise Dataset

reader = Reader(
    rating_scale=(
        svd_df["Rating"].min(),
        svd_df["Rating"].max()
    )
)

svd_data = Dataset.load_from_df(
    svd_df[
        [
            "AuthorId",
            "RecipeId",
            "Rating"
        ]
    ],
    reader
)

print("Surprise dataset created successfully.")

Surprise dataset created successfully.


In [31]:
# Train-Test Split

svd_trainset, svd_testset = train_test_split(
    svd_data,
    test_size=0.2,
    random_state=42
)

print("=" * 60)
print("SVD TRAIN TEST SPLIT")
print("=" * 60)

print(f"Training Users   : {svd_trainset.n_users:,}")
print(f"Training Recipes : {svd_trainset.n_items:,}")
print(f"Testing Ratings  : {len(svd_testset):,}")

SVD TRAIN TEST SPLIT
Training Users   : 17,502
Training Recipes : 36,454
Testing Ratings  : 84,114


### SVD Data Findings

The SVD model is trained using the filtered collaborative filtering dataset containing:

- 420,570 user-recipe rating interactions
- 17,546 active users
- 36,632 actively rated recipes

The dataset remains highly sparse, making direct similarity-based methods less effective. Matrix Factorization is suitable for this scenario because it learns compact latent representations instead of relying only on overlapping user or recipe interactions.

The train-test split preserves the majority of users and recipes in the training data, allowing the model to learn meaningful preference patterns while keeping unseen ratings for evaluation.

### Training the SVD Model

The SVD model factorizes the user-recipe rating matrix into latent user and recipe feature vectors.

The number of latent factors controls how many hidden preference dimensions the model learns.

In [32]:
# Train SVD Model

svd_model = SVD(
    n_factors=100,
    n_epochs=20,
    lr_all=0.005,
    reg_all=0.02,
    random_state=42
)

svd_model.fit(svd_trainset)

print("SVD model training completed.")

SVD model training completed.


In [33]:
# Generate Predictions on Test Data

svd_predictions = svd_model.test(
    svd_testset
)

svd_predictions[:5]

[Prediction(uid=212609, iid=238248, r_ui=5.0, est=4.749933251941331, details={'was_impossible': False}),
 Prediction(uid=621626, iid=391878, r_ui=5.0, est=4.59787480577433, details={'was_impossible': False}),
 Prediction(uid=176615, iid=75822, r_ui=5.0, est=4.949432897520715, details={'was_impossible': False}),
 Prediction(uid=611660, iid=112117, r_ui=5.0, est=4.599136607350793, details={'was_impossible': False}),
 Prediction(uid=611201, iid=143802, r_ui=5.0, est=4.905940865123103, details={'was_impossible': False})]

In [34]:
# Evaluate SVD Model

print("=" * 60)
print("SVD MODEL PERFORMANCE")
print("=" * 60)

svd_rmse = accuracy.rmse(
    svd_predictions
)

svd_mae = accuracy.mae(
    svd_predictions
)

SVD MODEL PERFORMANCE
RMSE: 0.9031
MAE:  0.5237


### SVD Model Evaluation

The trained SVD model was evaluated using Root Mean Squared Error (RMSE) and Mean Absolute Error (MAE).

#### Mean Absolute Error (MAE)

MAE measures the average absolute difference between actual user ratings and predicted ratings.

The achieved MAE score:

- MAE: 0.5237

This means that, on average, the SVD model predictions differ from the actual user ratings by approximately 0.52 rating points.

For a rating scale between 0 and 5, an average error close to half a rating point indicates that the model successfully captures user preference patterns.


#### Root Mean Squared Error (RMSE)

RMSE also measures prediction error but gives higher penalties to large mistakes.

The achieved RMSE score:

- RMSE: 0.9031

This indicates that most predictions remain within approximately one rating point of the user's actual rating.

Considering the extremely sparse nature of the dataset, this performance is acceptable for a collaborative filtering system.


The original interaction matrix contained very few observed user-recipe interactions compared to all possible combinations. SVD handles this sparsity by learning latent representations instead of depending on direct rating overlap.

The normalized SVD score (0-1 scale) will be used as the collaborative preference component in the final hybrid recommendation engine.

### Save the SVD Model

Training a matrix factorization model such as Singular Value Decomposition (SVD) can be computationally expensive, particularly for large interaction datasets. Therefore, once the model has been trained and evaluated, it is serialized and stored on disk for future use.

Saving the trained model eliminates the need to retrain it every time recommendations are generated. Instead, the saved model can be loaded directly by the application, significantly reducing startup time and computational overhead.

In this project, the trained SVD model is saved using the `joblib` library, producing a reusable model file that will later be loaded by the hybrid recommendation system during inference.

In [35]:
import os

os.makedirs("models", exist_ok=True)

joblib.dump(
    svd_model,
    "models/svd_model.pkl"
)

print("SVD model saved successfully.")

SVD model saved successfully.


### SVD Recommendations

The trained SVD model is used to estimate user preference for recipes that the user has not previously rated.

Unlike the evaluation phase, recommendation generation focuses on ranking recipes rather than predicting exact rating values.

By default, the Surprise library clips predicted ratings to the original rating scale (0–5). While this is appropriate for evaluating rating prediction accuracy, it can reduce the quality of recommendation ranking because multiple highly preferred recipes may receive the same maximum predicted rating.

For example, internal preference estimates such as:

- Recipe A → 5.42
- Recipe B → 5.37
- Recipe C → 5.18

would all be clipped to:

- Recipe A → 5.00
- Recipe B → 5.00
- Recipe C → 5.00

This removes useful ranking information between highly recommended recipes.

Since the objective of this recommender is to identify the recipes a user is most likely to enjoy rather than predict the exact rating they would assign, unclipped SVD predictions are used during recommendation generation (`clip=False`).

The resulting preference scores are then converted into percentile rankings.

The percentile-based SVD score has the following interpretation:

- **1.00** → Among the strongest collaborative recommendations for the user.
- **0.50** → Around the median recommendation.
- **0.00** → Among the weakest collaborative recommendations.

This produces a normalized collaborative filtering score between 0 and 1, making it directly compatible with the Content-Based Similarity Score and Nutrition Matching Score in the final hybrid recommendation system.

In [36]:
# SVD Recommendation Function

def recommend_svd(
    user_id,
    model,
    reviews_df,
    recipes_df,
    top_n=10
):
    """
    Generate personalized recipe recommendations
    using a trained SVD model.

    Parameters
    ----------
    user_id : int
        Target user.

    model : surprise.SVD
        Trained SVD model.

    reviews_df : pd.DataFrame
        Collaborative filtering review dataset.

    recipes_df : pd.DataFrame
        Recipe metadata.

    top_n : int, default=10
        Number of recommendations.

    Returns
    -------
    pd.DataFrame
        RecipeId
        Name
        PredictedRating
        SVDScore
    """

    # Recipes already rated by the user

    rated_recipes = set(
        reviews_df.loc[
            reviews_df["AuthorId"] == user_id,
            "RecipeId"
        ]
    )


    # Candidate recipes

    candidate_recipes = (
        set(reviews_df["RecipeId"])
        - rated_recipes
    )


    # Predict preference scores

    predictions = []

    for recipe_id in candidate_recipes:

        prediction = model.predict(
            uid=user_id,
            iid=recipe_id,
            clip=False
        )

        predictions.append(
            [
                recipe_id,
                prediction.est
            ]
        )


    recommendations = pd.DataFrame(
        predictions,
        columns=[
            "RecipeId",
            "PreferenceScore"
        ]
    )
    
        
    # Percentile-based collaborative score

    recommendations["SVDScore"] = (
        recommendations["PreferenceScore"]
        .rank(
            pct=True,
            method="average"
        )
    )


    # Select Top Recommendations

    recommendations = (
        recommendations
        .sort_values(
            by="SVDScore",
            ascending=False
        )
        .head(top_n)
    )


    # Add recipe names

    recommendations = recommendations.merge(
        recipes_df[
            [
                "RecipeId",
                "Name"
            ]
        ],
        on="RecipeId",
        how="left"
    )


    # Final Output

    recommendations = recommendations[
        [
            "RecipeId",
            "Name",
            "PreferenceScore",
            "SVDScore"
        ]
    ]


    return recommendations

In [37]:
# Example SVD Recommendations

sample_user = 37449

svd_recommendations = recommend_svd(
    user_id=sample_user,
    model=svd_model,
    reviews_df=cf_reviews_df,
    recipes_df=recipes_df,
    top_n=10
)

svd_recommendations

,RecipeId,Name,PreferenceScore,SVDScore
0,152161,Herb-Roasted Chicken,5.424550,1.000000
1,16259,Almond Chicken,5.418502,0.999971
2,83634,Bacon &amp; Egg Casserole,5.417078,0.999942
3,495275,Peppered Buffalo Ranch Shrimp Pizza #RSC,5.409642,0.999913
4,130244,Bacon and Egg Empanadas,5.384276,0.999884
5,135104,Italian Meatballs,5.353136,0.999855
6,16432,Panini Sandwiches,5.350933,0.999826
7,112454,Spicy Chicken and Black Bean Pasta,5.348281,0.999797
8,291635,Steel Cut Oats Ala Alton Brown,5.345305,0.999768
9,86192,Fluffy Morning Pancakes,5.344656,0.999739


### SVD Recommendation Findings

The SVD recommender successfully generates personalized recipe recommendations for recipes that the target user has not previously interacted with.

Unlike the evaluation phase, recommendation generation uses **unclipped preference estimates (`clip=False`)** instead of clipped rating predictions. This preserves the small differences between highly preferred recipes that would otherwise receive identical maximum ratings after clipping, allowing the model to produce a more meaningful ranking of candidate recipes.

The values shown in the **Predicted Preference** column represent the raw preference estimates produced by the trained SVD model. Since clipping is disabled during recommendation generation, these values may exceed the original rating scale and should therefore be interpreted as **relative preference estimates rather than actual predicted user ratings**.

To create a standardized collaborative filtering signal, the raw preference estimates are converted into **percentile-based SVD Scores**. This normalization transforms every prediction into a value between **0 and 1** while preserving the relative ordering of recipes.

It is important to note that the **SVD Score is a normalized ranking score and not a probability or predicted user rating**. An SVD Score of **1.00** simply indicates that the recipe is the **highest-ranked collaborative recommendation among all candidate recipes for the target user**. Likewise, lower scores indicate progressively weaker collaborative recommendations relative to the remaining candidate recipes.

The percentile-based SVD Score can therefore be interpreted as follows:

- **1.00** → Highest-ranked collaborative recommendation.
- **0.50** → Approximately the median collaborative recommendation.
- **0.00** → Among the lowest-ranked collaborative recommendations.

This normalization makes the collaborative filtering output directly comparable with the Content Similarity Score and Nutrition Matching Score, enabling all three components to be combined within the final hybrid recommendation system without differences in scale affecting the final recommendation ranking.

## Model Selection

Throughout this notebook, three collaborative filtering approaches were explored:

- User-Based Collaborative Filtering
- Item-Based Collaborative Filtering
- Singular Value Decomposition (SVD)

While all three methods generate personalized recommendations using historical user interactions, **Singular Value Decomposition (SVD)** was selected as the final collaborative filtering model because it addresses several limitations of traditional memory-based approaches while providing better scalability and generalization for large recommendation datasets.

### Why SVD Instead of User-Based Collaborative Filtering?

User-Based Collaborative Filtering recommends recipes based on users with similar historical preferences. Although this approach is intuitive and easy to understand, it suffers from several practical limitations:

- Finding similar users becomes increasingly expensive as the number of users grows.
- Recommendation quality depends heavily on the availability of sufficiently similar neighbours.
- Sparse interaction matrices often reduce the reliability of user similarity calculations.
- Performance deteriorates for users with relatively few interactions.

SVD overcomes these limitations by learning compact latent representations of users instead of repeatedly searching for neighbouring users during recommendation generation.

### Why SVD Instead of Item-Based Collaborative Filtering?

Item-Based Collaborative Filtering recommends recipes based on similarities between previously interacted recipes and candidate recipes.

Although item similarities are generally more stable than user similarities, this approach still relies entirely on explicit historical co-occurrence patterns.

As the number of recipes increases, computing and maintaining large item similarity matrices becomes computationally expensive. Furthermore, Item-Based Collaborative Filtering cannot effectively discover hidden relationships between recipes that have never been directly associated through user interactions.

SVD learns these hidden relationships by projecting both users and recipes into a shared latent feature space, enabling recommendations beyond direct neighbour similarity.

### Why SVD Was Selected

Compared to the two memory-based approaches, SVD offers several advantages for this project:

- Learns latent user preferences and hidden recipe characteristics instead of relying solely on explicit neighbour similarities.
- Handles sparse user–recipe interaction data more effectively through matrix factorization.
- Generalizes well to unseen user–recipe combinations by estimating preference scores from learned latent factors.
- Scales efficiently for large datasets containing thousands of users and recipes because recommendations are generated using learned latent vectors rather than repeated similarity computations.
- Produces a normalized collaborative filtering signal that integrates naturally with the Content-Based Similarity Score and Nutrition Matching Score in the final hybrid recommendation system.

### Evaluation vs Recommendation

Two different objectives were considered during this notebook.

During **model evaluation**, predictions were clipped to the original rating scale (0–5) because the objective was to measure rating prediction accuracy using RMSE and MAE.

During **recommendation generation**, unclipped preference estimates were used because the objective shifted from predicting exact ratings to ranking recipes according to user preference. These raw preference estimates were subsequently converted into percentile-based collaborative scores between **0 and 1**, preserving the relative ordering of recommendations while providing a normalized collaborative signal for the hybrid recommendation system.

### Final Outcome

Based on the comparative analysis of all three collaborative filtering techniques, **Singular Value Decomposition (SVD)** was selected as the collaborative recommendation model for this project.

The model effectively captures latent user preferences, generates personalized recommendations for unseen recipes, and produces a scalable collaborative filtering signal that will be combined with content-based similarity and nutrition matching in the final hybrid recommendation engine.